# Phase 4.1 — main model with slower training (lr 1e-4, dropout 0.2) on Colab GPU

**Before running:** Runtime → Change runtime type → **L4 GPU**. Then Runtime → **Run all** and keep the tab open
(roughly 15–45 minutes; the run stops early when the inner hold-out stops improving).

**One change, fixed in advance (not a search):** the v1 main model stopped at epochs 4 / 2 / 1 with lr 1e-3, so it overfits fast.
This run uses the PatchTST paper defaults, **lr 1e-4 and dropout 0.2** (Nie et al., 2023). To give slower learning room,
max epochs goes from 15 to 30 and early-stopping patience from 3 to 10.
Everything else (model `multiscale_sector` + news, seeds 20261015/16/17, look-back 60, data, splits) is identical to the
v1 run `main_model_v1_news_r2`, so the two compare like for like.

What counts as fixed: best epochs well past 4, and confusion-grid rows that separate by more than v1's +2 points.

Outputs in `MyDrive/Senior/results/`: `main_model_v1_news_p41.json` (scores, grids, settings, per-epoch history),
`main_model_v1_news_p41_preds.csv.gz` (every prediction, per seed) and `results_summary_p41.xlsx` (v1 vs Phase 4.1).
Validation block only; the test block stays sealed.


In [1]:
!nvidia-smi -L

GPU 0: NVIDIA L4 (UUID: GPU-1cffe3a2-f5d8-0519-2be2-27a94dcbbbcc)


In [2]:
from google.colab import drive
drive.mount('/content/drive')
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, OUT_DIR = f'{SENIOR}/prices', f'{SENIOR}/results'

Mounted at /content/drive


In [3]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
import hashlib
for f in ['scripts/baselines_deep.py', 'scripts/summarize_results.py']:
    print(f, hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12])
src = open('scripts/baselines_deep.py', encoding='utf-8').read()
assert '--dropout' in src and 'SET["dropout"]' in src, 'stop: push the latest baselines_deep.py from your PC (with the --lr/--dropout/--patience flags), then run again'


/content/Senior-Project
ffd950a (HEAD -> main, origin/main, origin/HEAD) Add Phase 4.1 training config and CLI overrides
scripts/baselines_deep.py f1689e0b9020
scripts/summarize_results.py e9618ff13de3


In [4]:
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo "$OUT_DIR"
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
# Files must match the PC copies (hashes recorded 2026-10-15; sentiment file as stored in git, LF line endings)
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the PC copy'

OK  data/raw/prices/siamchart/set50_union_2019_2025.csv
OK  data/raw/prices/yahoo/yahoo_events_2019_2025.csv
OK  data/derived/sentiment_scores.csv


In [5]:
!python tests/test_pipeline.py

PASS dividend adjustment removes the XD drop (raw -0.0216 -> adj 0.0011)
PASS split days give normal returns after adjustment (max |r| 0.172)
PASS no return across the TRUE changeover
PASS no return across the SCB changeover
PASS no return across the GULF changeover
PASS no return across the TIDLOR changeover
PASS unexplained GULF 2020-04-16 jump is a break
PASS point-in-time membership: 40-50 members with prices on >95% of days
PASS indicators at t ignore all data after t
PASS fwd_5 at t = sum of returns t+1..t+5
PASS forward returns stop at a changeover
PASS classify has no default sigma or k
PASS fit_k balances classes (k=0.432, {'down': 0.3381, 'flat': 0.3317, 'up': 0.3301})
PASS window = the 60 rows ending at t (inclusive), nothing after t
PASS window never reaches across a changeover
PASS sector table covers all 70 union tickers
PASS blocks are chronological and disjoint
PASS 10 purged dates between train and val
PASS scaler refuses to fit on non-training rows
all pipeline tests 

In [6]:
# Phase 4.1: main model WITH news, lr 1e-4, dropout 0.2 (all other settings as in v1)
!python scripts/baselines_deep.py --models multiscale_sector --news data/derived/sentiment_scores.csv \
    --lr 1e-4 --dropout 0.2 --max-epochs 30 --patience 10 \
    --out "$OUT_DIR/main_model_v1_news_p41.json"


settings: lr=0.0001, dropout=0.2, max_epochs=30, patience=10
rows: fit 44060 / inner-ES 5050 / val 10277; window 60 x 21
    multiscale_sector seed 20261015 ep 1: loss 2.1402  inner-ES macro-F1 0.3387  (12s)
    multiscale_sector seed 20261015 ep 2: loss 2.1094  inner-ES macro-F1 0.3615  (11s)
    multiscale_sector seed 20261015 ep 3: loss 2.0719  inner-ES macro-F1 0.3579  (11s)
    multiscale_sector seed 20261015 ep 4: loss 2.0339  inner-ES macro-F1 0.3579  (11s)
    multiscale_sector seed 20261015 ep 5: loss 1.9968  inner-ES macro-F1 0.3636  (11s)
    multiscale_sector seed 20261015 ep 6: loss 1.9547  inner-ES macro-F1 0.3668  (11s)
    multiscale_sector seed 20261015 ep 7: loss 1.9231  inner-ES macro-F1 0.3632  (11s)
    multiscale_sector seed 20261015 ep 8: loss 1.8882  inner-ES macro-F1 0.3652  (11s)
    multiscale_sector seed 20261015 ep 9: loss 1.8572  inner-ES macro-F1 0.3608  (11s)
    multiscale_sector seed 20261015 ep 10: loss 1.8212  inner-ES macro-F1 0.3654  (11s)
    mult

In [7]:
# Did it train longer? Best epoch per seed (v1 was 4 / 2 / 1) and the settings actually used
import json
r = json.load(open(f'{OUT_DIR}/main_model_v1_news_p41.json'))
print({k: r['settings'][k] for k in ('lr', 'dropout', 'max_epochs', 'patience')}, r['git_commit'], r['gpu'], r['torch'])
for run in r['models']['multiscale_sector']['runs']:
    print('seed', run.get('seed'), 'best epoch', run.get('best_epoch'), 'of', len(run.get('history', [])),
          '| val macro-F1', {h: round(run['val'][h]['macro_f1'], 3) for h in ('h1', 'h5', 'h10')})


{'lr': 0.0001, 'dropout': 0.2, 'max_epochs': 30, 'patience': 10} ffd950a NVIDIA L4 2.11.0+cu130
seed 20261015 best epoch 6 of 16 | val macro-F1 {'h1': 0.339, 'h5': 0.335, 'h10': 0.345}
seed 20261016 best epoch 2 of 12 | val macro-F1 {'h1': 0.304, 'h5': 0.347, 'h10': 0.344}
seed 20261017 best epoch 5 of 15 | val macro-F1 {'h1': 0.337, 'h5': 0.35, 'h10': 0.335}


In [8]:
# Side-by-side spreadsheet: v1 (lr 1e-3) vs Phase 4.1 (Claude reads this from Drive)
!python scripts/summarize_results.py --inputs "$OUT_DIR/main_model_v1_news_r2.json" "$OUT_DIR/main_model_v1_news_p41.json" --out "$OUT_DIR/results_summary_p41.xlsx"


wrote /content/drive/MyDrive/Senior/results/results_summary_p41.xlsx

                model_label horizon  seeds  macro_f1_mean  macro_f1_std  accuracy_mean
Multi-Scale PatchTST + news      h1      3         0.3298        0.0085         0.3377
Multi-Scale PatchTST + news      h5      3         0.3405        0.0118         0.3440
Multi-Scale PatchTST + news     h10      3         0.3401        0.0104         0.3437
Multi-Scale PatchTST + news      h1      3         0.3267        0.0197         0.3356
Multi-Scale PatchTST + news      h5      3         0.3439        0.0081         0.3452
Multi-Scale PatchTST + news     h10      3         0.3415        0.0054         0.3433

Multi-Scale PatchTST + news  h1  [main_model_v1_news_r2.json]  (rows = actual, % of row; all seeds)
actual  pct_down  pct_flat  pct_up  row_total
  down      36.4      43.0    20.5      10527
  flat      38.5      42.0    19.4      10563
    up      37.7      40.4    21.9       9741

Multi-Scale PatchTST + news  h5  [m